## Part 4.1: Initialise PostgreSQL Schemas, Tables, Constraints, and Load Session

### Objective

Prepare the PostgreSQL database structures required for controlled Cyclistic data loading.

Parts 1–3 validated the environment, established the approved schema contract, and completed the data-quality audit. Part 4 begins database processing.

This section creates database structures only. It does not load CSV records into staging or quarantine.

### Database Architecture

The initial database-processing layer uses three PostgreSQL schemas:

```text
stg
quarantine
audit
```

Their responsibilities are:

| Schema       | Responsibility                                                                                           |
| ------------ | -------------------------------------------------------------------------------------------------------- |
| `stg`        | Stores source records that pass failure-severity quality controls and remain eligible for transformation |
| `quarantine` | Stores source records that violate failure-severity rules                                                |
| `audit`      | Stores database-load execution history and reconciliation information                                    |

PostgreSQL schemas must exist before schema-qualified tables can be created.

The pipeline therefore creates the schemas explicitly:

```sql
CREATE SCHEMA IF NOT EXISTS stg;
CREATE SCHEMA IF NOT EXISTS quarantine;
CREATE SCHEMA IF NOT EXISTS audit;
```

PostgreSQL and Azure Database for PostgreSQL do not automatically convert:

```text
stg.cyclistic_rides
```

into:

```text
stg_cyclistic_rides
```

The first form refers to table `cyclistic_rides` inside schema `stg`. The second form is a separate table name in the current schema.

### Staging Table

`stg.cyclistic_rides` stores records eligible for downstream processing.

It contains the approved Cyclistic source fields:

* `ride_id`;
* `rideable_type`;
* `started_at`;
* `ended_at`;
* station names and identifiers;
* start and end coordinates;
* `member_casual`.

It also adds technical lineage fields:

* `load_session_id`;
* `quality_audit_session_id`;
* `source_file`;
* `source_row_number`;
* `source_record_key`;
* `ride_duration_seconds`;
* `has_quality_warning`;
* `warning_rule_ids`;
* `loaded_at`.

The source record key uniquely identifies a physical source row within one load session.

### Quarantine Table

`quarantine.cyclistic_rides` preserves records that violate failure-severity quality rules.

It stores:

* the original source values;
* source filename and row number;
* database-load and quality-audit session IDs;
* failed rule IDs;
* quarantine reason;
* optional warning rule IDs;
* quarantine timestamp.

The initial expected quarantine condition is:

```text
DQ007_END_AFTER_START
```

for the 29 records in `202511-divvy-tripdata.csv` whose ending timestamp is not later than their starting timestamp.

No source record is deleted or corrected in this section.

### Pipeline Job Log

`audit.pipeline_job_log` records one row for each database-load execution.

The initial record is inserted with status:

```text
INITIALISED
```

Later Part 4 sections will update the same record through states such as:

```text
LOADING
VALIDATING
COMPLETED
FAILED
```

The table records:

* load session ID;
* related quality-audit session ID;
* schema and ruleset versions;
* source file and row counts;
* staging and quarantine counts;
* rejected and warning counts;
* execution timestamps;
* error information;
* execution metadata.

### Transaction Policy

Schema creation, table creation, index creation, validation, and load-session registration are executed in a PostgreSQL transaction.

If a database operation fails:

* the transaction is rolled back;
* no partial job-log record is committed;
* the failure is written to `pipeline_audit.log`;
* the notebook raises the original error.

### Idempotency Policy

The database objects use:

```sql
CREATE SCHEMA IF NOT EXISTS
CREATE TABLE IF NOT EXISTS
CREATE INDEX IF NOT EXISTS
```

This allows the setup cell to be rerun safely.

However, `IF NOT EXISTS` does not prove that an existing table has the correct columns. The section therefore validates required table columns after creation and stops if an incompatible existing table is detected.

### Expected Outcome

* PostgreSQL connectivity is reverified.
* The current database and role are identified.
* Schema-creation permission is confirmed.
* `stg`, `quarantine`, and `audit` exist.
* Required tables and indexes exist.
* Existing table structures contain all required columns.
* A unique database-load session is generated.
* The load session is registered as `INITIALISED`.
* Part 4.2 can begin controlled chunked loading and row routing.


In [ ]:
# ==========================================
# Part 4.1: Initialise PostgreSQL Schemas
# ==========================================

from datetime import datetime
from pathlib import Path
from uuid import uuid4

import psycopg2
from psycopg2 import OperationalError, sql


# ------------------------------------------
# Validate required dependencies
# ------------------------------------------

REQUIRED_PART_4_1_VALUES = {
    "DB_HOST": DB_HOST,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_PORT": DB_PORT,
    "DB_SSLMODE": DB_SSLMODE,
    "PIPELINE_AUDIT_PATH": PIPELINE_AUDIT_PATH,
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
}

missing_part_4_1_values = [
    variable_name
    for variable_name, variable_value
    in REQUIRED_PART_4_1_VALUES.items()
    if (
        variable_value is None
        or (
            isinstance(variable_value, str)
            and not variable_value.strip()
        )
    )
]

if missing_part_4_1_values:
    raise RuntimeError(
        "Part 4.1 is missing required value(s): "
        + ", ".join(
            sorted(missing_part_4_1_values)
        )
    )


if not isinstance(
    PIPELINE_AUDIT_PATH,
    Path,
):
    raise TypeError(
        "PIPELINE_AUDIT_PATH must be a pathlib.Path object."
    )


if not PIPELINE_AUDIT_PATH.exists():
    raise FileNotFoundError(
        "Pipeline audit log does not exist:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )


if not PIPELINE_AUDIT_PATH.is_file():
    raise FileExistsError(
        "PIPELINE_AUDIT_PATH exists but is not a file:\n"
        f"{PIPELINE_AUDIT_PATH}"
    )


# ------------------------------------------
# Initialise database-processing session
# ------------------------------------------

# Reuse this identifier throughout Parts 4.1–4.5.
DATABASE_PROCESSING_SESSION_ID = (
    uuid4().hex[:8]
)

DATABASE_PROCESSING_START_TIME = (
    datetime.now(
        MELBOURNE_TIMEZONE
    )
)

database_run_warnings = []
database_run_errors = []


SUPPORTED_DATABASE_LOG_LEVELS = {
    "INFO",
    "WARNING",
    "ERROR",
    "CRITICAL",
}


def log_database_event(
    message: str,
    level: str = "INFO",
) -> str:
    """
    Append one database-processing event to the shared pipeline
    audit log and print it in the notebook.
    """
    normalised_level = level.upper().strip()

    if (
        normalised_level
        not in SUPPORTED_DATABASE_LOG_LEVELS
    ):
        raise ValueError(
            "Unsupported database log level: "
            f"{level}"
        )

    if (
        not isinstance(message, str)
        or not message.strip()
    ):
        raise ValueError(
            "Database log message must be a non-empty string."
        )

    timestamp = datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds")

    log_entry = (
        f"[{timestamp}] "
        f"[DB_RUN:{DATABASE_PROCESSING_SESSION_ID}] "
        f"[{normalised_level}] "
        f"{message.strip()}"
    )

    with PIPELINE_AUDIT_PATH.open(
        mode="a",
        encoding="utf-8",
    ) as audit_file:
        audit_file.write(
            log_entry + "\n"
        )

    print(log_entry)

    event_record = {
        "timestamp": timestamp,
        "database_processing_session_id": (
            DATABASE_PROCESSING_SESSION_ID
        ),
        "level": normalised_level,
        "message": message.strip(),
    }

    if normalised_level == "WARNING":
        database_run_warnings.append(
            event_record
        )

    elif normalised_level in {
        "ERROR",
        "CRITICAL",
    }:
        database_run_errors.append(
            event_record
        )

    return log_entry


# ------------------------------------------
# Define required PostgreSQL schemas
# ------------------------------------------

REQUIRED_DATABASE_SCHEMAS = (
    "stg",
    "quarantine",
    "audit",
)


# ------------------------------------------
# Open transaction and initialise schemas
# ------------------------------------------

connection = None
cursor = None

schemas_existing_before = []
schemas_created_this_run = []
verified_schemas = []


log_database_event(
    message=(
        "DATABASE_PROCESSING_SESSION_START: "
        "PostgreSQL database processing initialised."
    ),
    level="INFO",
)


try:
    connection = psycopg2.connect(
        host=DB_HOST,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        port=DB_PORT,
        sslmode=DB_SSLMODE,
        connect_timeout=10,
    )

    connection.autocommit = False
    cursor = connection.cursor()


    # --------------------------------------
    # Confirm connection identity
    # --------------------------------------

    cursor.execute(
        """
        SELECT
            current_database(),
            current_user;
        """
    )

    (
        connected_database,
        connected_user,
    ) = cursor.fetchone()


    if connected_database != DB_NAME:
        raise RuntimeError(
            "Connected database does not match DB_NAME. "
            f"Expected={DB_NAME}; "
            f"Observed={connected_database}."
        )


    log_database_event(
        message=(
            "DATABASE_CONNECTION_OPENED: "
            f"database={connected_database}; "
            f"user={connected_user}; "
            f"host={DB_HOST}; "
            f"port={DB_PORT}."
        ),
        level="INFO",
    )


    # --------------------------------------
    # Inspect schemas before initialisation
    # --------------------------------------

    cursor.execute(
        """
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name = ANY(%s)
        ORDER BY schema_name;
        """,
        (list(REQUIRED_DATABASE_SCHEMAS),),
    )

    schemas_existing_before = [
        row[0]
        for row in cursor.fetchall()
    ]


    schemas_missing_before = [
        schema_name
        for schema_name
        in REQUIRED_DATABASE_SCHEMAS
        if schema_name
        not in schemas_existing_before
    ]


    # --------------------------------------
    # Validate database CREATE privilege
    # --------------------------------------

    cursor.execute(
        """
        SELECT has_database_privilege(
            current_user,
            current_database(),
            'CREATE'
        );
        """
    )

    can_create_schema = bool(
        cursor.fetchone()[0]
    )


    if (
        schemas_missing_before
        and not can_create_schema
    ):
        raise PermissionError(
            "The connected PostgreSQL role cannot create "
            "schemas in the current database. "
            "Missing schema(s): "
            + ", ".join(
                schemas_missing_before
            )
        )


    # --------------------------------------
    # Create schemas idempotently
    # --------------------------------------

    for schema_name in (
        REQUIRED_DATABASE_SCHEMAS
    ):
        create_schema_statement = (
            sql.SQL(
                "CREATE SCHEMA IF NOT EXISTS {}"
            ).format(
                sql.Identifier(schema_name)
            )
        )

        cursor.execute(
            create_schema_statement
        )


        if schema_name in schemas_existing_before:
            log_database_event(
                message=(
                    "DATABASE_SCHEMA_FOUND: "
                    f"schema={schema_name}."
                ),
                level="INFO",
            )

        else:
            schemas_created_this_run.append(
                schema_name
            )

            log_database_event(
                message=(
                    "DATABASE_SCHEMA_CREATED: "
                    f"schema={schema_name}."
                ),
                level="INFO",
            )


    # --------------------------------------
    # Verify schema existence and ownership
    # --------------------------------------

    cursor.execute(
        """
        SELECT
            namespace.nspname AS schema_name,
            owner_role.rolname AS schema_owner
        FROM pg_namespace AS namespace
        INNER JOIN pg_roles AS owner_role
            ON owner_role.oid = namespace.nspowner
        WHERE namespace.nspname = ANY(%s)
        ORDER BY namespace.nspname;
        """,
        (list(REQUIRED_DATABASE_SCHEMAS),),
    )

    verified_schema_rows = (
        cursor.fetchall()
    )

    verified_schema_metadata = {
        schema_name: {
            "owner": schema_owner,
        }
        for (
            schema_name,
            schema_owner,
        ) in verified_schema_rows
    }


    missing_after_initialisation = [
        schema_name
        for schema_name
        in REQUIRED_DATABASE_SCHEMAS
        if schema_name
        not in verified_schema_metadata
    ]


    if missing_after_initialisation:
        raise RuntimeError(
            "Required schema verification failed. "
            "Missing schema(s): "
            + ", ".join(
                missing_after_initialisation
            )
        )


    # --------------------------------------
    # Verify schema privileges
    # --------------------------------------

    for schema_name in (
        REQUIRED_DATABASE_SCHEMAS
    ):
        cursor.execute(
            """
            SELECT
                has_schema_privilege(
                    current_user,
                    %s,
                    'USAGE'
                ),
                has_schema_privilege(
                    current_user,
                    %s,
                    'CREATE'
                );
            """,
            (
                schema_name,
                schema_name,
            ),
        )

        (
            has_usage_privilege,
            has_create_privilege,
        ) = cursor.fetchone()


        if not has_usage_privilege:
            raise PermissionError(
                "The connected PostgreSQL role does not have "
                f"USAGE privilege on schema '{schema_name}'."
            )


        if not has_create_privilege:
            raise PermissionError(
                "The connected PostgreSQL role does not have "
                f"CREATE privilege on schema '{schema_name}'."
            )


        verified_schemas.append(
            {
                "schema_name": schema_name,
                "owner": (
                    verified_schema_metadata[
                        schema_name
                    ]["owner"]
                ),
                "usage_privilege": bool(
                    has_usage_privilege
                ),
                "create_privilege": bool(
                    has_create_privilege
                ),
            }
        )

        log_database_event(
            message=(
                "DATABASE_SCHEMA_VERIFIED: "
                f"schema={schema_name}; "
                f"owner="
                f"{verified_schema_metadata[schema_name]['owner']}; "
                "usage_privilege=True; "
                "create_privilege=True."
            ),
            level="INFO",
        )


    # --------------------------------------
    # Commit schema initialisation
    # --------------------------------------

    connection.commit()

    log_database_event(
        message=(
            "DATABASE_SCHEMA_INITIALISATION_COMMITTED: "
            f"created={len(schemas_created_this_run)}; "
            f"existing={len(schemas_existing_before)}; "
            f"verified={len(verified_schemas)}."
        ),
        level="INFO",
    )


except OperationalError as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_SCHEMA_INITIALISATION_FAILED: "
            "Unable to connect to PostgreSQL. "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise ConnectionError(
        "Unable to open the PostgreSQL connection required "
        "for Part 4.1."
    ) from error


except (
    PermissionError,
    RuntimeError,
    psycopg2.DatabaseError,
) as error:
    if connection is not None:
        connection.rollback()

    log_database_event(
        message=(
            "DATABASE_SCHEMA_INITIALISATION_ROLLED_BACK: "
            f"error_type={type(error).__name__}; "
            f"error={error}"
        ),
        level="CRITICAL",
    )

    raise


finally:
    if cursor is not None:
        cursor.close()
        print("✓ Database cursor closed.")

    if connection is not None:
        connection.close()
        print("✓ Database connection closed.")


# ------------------------------------------
# Build Part 4.1 summary
# ------------------------------------------

DATABASE_SCHEMA_INITIALISATION_SUMMARY = {
    "database_processing_session_id": (
        DATABASE_PROCESSING_SESSION_ID
    ),
    "database": connected_database,
    "connected_user": connected_user,
    "required_schemas": list(
        REQUIRED_DATABASE_SCHEMAS
    ),
    "schemas_existing_before": list(
        schemas_existing_before
    ),
    "schemas_created_this_run": list(
        schemas_created_this_run
    ),
    "verified_schemas": list(
        verified_schemas
    ),
    "status": "SUCCESS",
    "completed_at": datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds"),
}


# ------------------------------------------
# Display final summary
# ------------------------------------------

print(
    "\n========== PostgreSQL Schema Initialisation "
    "==========\n"
)

print(
    f"Database-processing session : "
    f"{DATABASE_PROCESSING_SESSION_ID}"
)
print(
    f"Database                    : "
    f"{connected_database}"
)
print(
    f"Connected role              : "
    f"{connected_user}"
)
print(
    f"Required schemas            : "
    f"{len(REQUIRED_DATABASE_SCHEMAS)}"
)
print(
    f"Schemas created             : "
    f"{len(schemas_created_this_run)}"
)
print(
    f"Schemas already existing    : "
    f"{len(schemas_existing_before)}"
)
print(
    f"Schemas verified            : "
    f"{len(verified_schemas)}"
)


print("\nSchema details:")

for schema_record in verified_schemas:
    print(
        f"  {schema_record['schema_name']:<12} "
        f"owner={schema_record['owner']:<20} "
        "USAGE=True CREATE=True"
    )


print("\n" + "=" * 75)
print("✓ Required PostgreSQL schemas exist.")
print("✓ Existing schemas were preserved.")
print("✓ Missing schemas were created idempotently.")
print("✓ Schema permissions verified.")
print("✓ Schema initialisation transaction committed.")
print("✓ Part 4.1 completed successfully.")
print("=" * 75)